# Soft arm model comparison

Standalone notebook: runs on Google Colab (or any Python ≥ 3.10) with no files from the project
repository. It installs the simulator package, defines every model with its fitted parameters, and
compares each model against the simulator under the same pressure commands.

- **Simulator:** `soft-robotic-arm` 0.4.2 from PyPI, `make_calibrated_sim()` (100 Hz control), treated as the robot.
- **Models:** One-mode, One-mode + gravity, M2, M3, F (defined in §3).
- **Test:** every model gets the same commands as the simulator and runs open loop from rest (free run).

Run all cells top to bottom (Runtime → Run all). The full run takes a few minutes; most of it is the
simulator.

## 1 — Install

In [ ]:
import importlib.util, subprocess, sys

NEEDED = {"soft_robotic_arm": "soft-robotic-arm==0.4.2", "scipy": "scipy"}   # Colab already has scipy
missing = [pkg for mod, pkg in NEEDED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
import soft_robotic_arm
print("soft-robotic-arm", getattr(soft_robotic_arm, "__version__", "installed"))

## 2 — Imports, constants and plot style

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy.signal import cont2discrete
from scipy.optimize import curve_fit
from IPython.display import HTML, display
from soft_robotic_arm import make_calibrated_sim

HZ, DT = 100.0, 0.01          # control rate (Hz) and step (s)
BASE = 4.5                    # settled command on every input, psi
SETTLE_S = 5.0                # simulator settles this long at BASE before each test

# Plot style: each model has its own colour, line style and marker (no green: red-green safe).
BLACK, ORANGE, SKY, BLUE = "#000000", "#E69F00", "#56B4E9", "#0072B2"
VERMILLION, PURPLE = "#D55E00", "#CC79A7"
SOLID, DASHED, DASHDOT = "-", (0, (5, 2.5)), (0, (6, 2, 1.5, 2))
DOTTED, LONGDASH = (0, (1.2, 1.8)), (0, (11, 3))
STYLE = {  # name -> (colour, line style, marker)
    "One-mode": (VERMILLION, DOTTED, "s"),
    "One-mode + gravity": (PURPLE, LONGDASH, "D"),
    "M2": (BLUE, SOLID, "o"),
    "M3": (ORANGE, DASHED, "^"),
    "F": (SKY, DASHDOT, "v"),
    "S2": (ORANGE, SOLID, "o"), "S3": (SKY, DASHED, "s"), "S4": (PURPLE, DASHDOT, "^"),
}
SIM_STYLE = dict(color=BLACK, ls="-", lw=3.2, alpha=0.85, label="Simulator")
WIDTH, PANEL_H = 12.0, 4.6

matplotlib.rcParams.update({
    "figure.dpi": 110, "font.size": 12, "axes.titlesize": 15, "axes.labelsize": 13,
    "legend.fontsize": 11, "lines.linewidth": 2.2, "axes.grid": True, "grid.alpha": 0.3,
})


def line(name: str, n: int, **kw) -> dict:
    c, ls, mk = STYLE[name]
    out = dict(color=c, ls=ls, marker=mk, label=name, markersize=7, markevery=max(1, n // 12))
    out.update(kw)
    return out

## 3 — Inputs, excitations and the models

**Inputs.** The arm has four pressure columns S1–S4 that push the tip toward +x, +y, −x, −y. On
`make_calibrated_sim()` S1 is a sealed reservoir, so the robot has three inputs: the commanded
pressures of S2, S3, S4 in psi (kept within 0–9). Below, u = command − 4.5 psi, and y = tip position
minus its settled value, in mm. The per-axis models take plane inputs ν<sub>x</sub> = −u<sub>3</sub>,
ν<sub>y</sub> = u<sub>2</sub> − u<sub>4</sub>.

**Excitation.** One test input: a pressure trace for each of the three inputs, played at the same
time. The traces differ on purpose: S2 and S4 push in opposite directions, so equal pressures cancel
in bending.

**Actuator** (M2, M3; F has its own delay): one first-order ODE plus a pure delay per input,

τ ṗ = −p + G u(t − T),   T = 0.5 s dead time, τ = 0.6 s lag, G ≈ 1.1,

where p is the pouch pressure above its settled value (psi).

| Model | Equations | Outputs | Delay + lag |
|---|---|---|---|
| One-mode | ÿ + c ẏ + k y = b ν per axis, k = 10.60 s⁻², c = 9.231 s⁻¹ | x, y | no |
| One-mode + gravity | same, k = 51.45 s⁻², c = 9.409 s⁻¹ | x, y | no |
| M2 | actuator, then ẏ = −a y + a (H p) per axis | x, y, z | yes |
| M3 | actuator, then ÿ + 2ζω ẏ + ω² y = ω² (H p) per axis | x, y, z | yes |
| F | state space per axis (A, B, C), input ν delayed | x, y | yes |

H (3×3, mm/psi) maps the three pouch pressures to x, y, z. All fitted values are in `PARAMS`.

In [ ]:
PARAMS = {
    "one_mode": {"k": 10.602415303139802, "c": 9.230907320069726, "b": -0.08203667215669287},
    "one_mode_gravity": {"k": 51.45086866273482, "c": 9.408628750381794, "b": -0.07903934655742169},
    "actuator": {"tau_s": 0.5994717450816, "gain": [1.1198163558706276, 1.0963542905121555, 1.1640064992186534], "delay_samples": 50},
    "M2": {"a": [4.7686307018369645, 4.69917138863036, 15.49350381103231],
           "H": [[-0.0032898079827158163, -1.8121363462879294, -0.0019863475369650183],
                 [1.8230768645607383, -0.0017432862629133616, -1.8133464514495357],
                 [-0.4928514516437697, -0.44608442864405806, -0.4895769295081067]]},
    "M3": {
        "x": {"wn": 7.610436118374175, "zeta": 0.7278983189895342, "H": [-0.00607923224312278, -1.7754834381101559, -0.003040663263496322]},
        "y": {"wn": 7.867160626614462, "zeta": 0.7721211236223006, "H": [1.7944185514093962, -0.000697655425465945, -1.798712943553855]},
        "z": {"wn": 31.658518909487213, "zeta": 0.8622431841362171, "H": [-0.49173486370802655, -0.4452106178515533, -0.4887877583480274]},
    },
    "F": {
        "x": {"delay_samples": 50,
              "A": [[-1.6681854273475776, 0.0, 0.0],
                    [0.0, 0.0, 1.0],
                    [1.0, -56.05921171996771, -10.301255828076245]],
              "B": [1.8295078778060205, 0.0, 0.0],
              "C": [0.0, 0.10001009529634684, 0.0]},
        "y": {"delay_samples": 50,
              "A": [[-1.668232547930015, 0.0, 0.0, 0.0],
                    [0.0, 0.0, 1.0, 0.0],
                    [0.0, 0.0, 0.0, 1.0],
                    [1.0, -74.68378221363929, -61.091005324887426, -9.8553136798169]],
              "B": [1.9047679806069808, 0.0, 0.0, 0.0],
              "C": [0.0, 0.1350157532233439, 0.07965478955129147, 0.0]},
    },
}

In [ ]:
def lsim(A, B, C, u: np.ndarray, delay: int = 0) -> np.ndarray:
    # ZOH-discretised x' = A x + B u(t - delay*DT), y = C x, from rest; y after each step.
    A, B, C = np.atleast_2d(A), np.atleast_2d(B), np.atleast_2d(C)
    Ad, Bd, *_ = cont2discrete((A, B, C, np.zeros((C.shape[0], B.shape[1]))), DT)
    ud = np.vstack([np.zeros((delay, u.shape[1])), u])[: len(u)]
    x, y = np.zeros(A.shape[0]), np.empty((len(u), C.shape[0]))
    for k in range(len(u)):
        x = Ad @ x + Bd @ ud[k]
        y[k] = C @ x
    return y


def plane_inputs(u: np.ndarray) -> np.ndarray:
    # (n, 2) plane inputs (x, y) from u = (u2, u3, u4): (-u3, u2 - u4)
    return np.column_stack([-u[:, 1], u[:, 0] - u[:, 2]])


def simulate_one_mode(u: np.ndarray, gravity: bool = False) -> np.ndarray:
    p = PARAMS["one_mode_gravity" if gravity else "one_mode"]
    k, c, b = p["k"], p["c"], abs(p["b"])            # b in m/psi, sign corrected
    A, B, C = [[0.0, 1.0], [-k, -c]], [[0.0], [b]], [[1.0, 0.0]]
    nu = plane_inputs(u)
    out = np.full((len(u), 3), np.nan)                 # no z output
    for i in (0, 1):
        out[:, i] = 1000.0 * lsim(A, B, C, nu[:, [i]])[:, 0]
    return out


def actuator_blocks():
    act = PARAMS["actuator"]
    return act["tau_s"], np.array(act["gain"]), act["delay_samples"]


def simulate_m2(u: np.ndarray) -> np.ndarray:
    tau, g, nd = actuator_blocks()
    a, H = np.array(PARAMS["M2"]["a"]), np.array(PARAMS["M2"]["H"])
    O = np.zeros((3, 3))
    A = np.block([[-np.diag(a), np.diag(a) @ H], [O, -np.eye(3) / tau]])
    B = np.vstack([O, np.diag(g) / tau])
    C = np.hstack([np.eye(3), O])
    return lsim(A, B, C, u, delay=nd)


def simulate_m3(u: np.ndarray) -> np.ndarray:
    tau, g, nd = actuator_blocks()
    w = np.array([PARAMS["M3"][a]["wn"] for a in "xyz"])
    z = np.array([PARAMS["M3"][a]["zeta"] for a in "xyz"])
    H = np.array([PARAMS["M3"][a]["H"] for a in "xyz"])
    O = np.zeros((3, 3))
    A = np.block([[O, np.eye(3), O],
                  [-np.diag(w**2), -np.diag(2 * z * w), np.diag(w**2) @ H],
                  [O, O, -np.eye(3) / tau]])
    B = np.vstack([O, O, np.diag(g) / tau])
    C = np.hstack([np.eye(3), O, O])
    return lsim(A, B, C, u, delay=nd)


def simulate_f(u: np.ndarray) -> np.ndarray:
    nu = plane_inputs(u)
    out = np.full((len(u), 3), np.nan)
    for i, ax in enumerate(("x", "y")):
        p = PARAMS["F"][ax]
        A, B, C = np.array(p["A"]), np.array(p["B"])[:, None], np.array(p["C"])[None, :]
        out[:, i] = 1000.0 * lsim(A, B, C, nu[:, [i]], delay=int(p["delay_samples"]))[:, 0]
    return out


MODELS = ("One-mode", "One-mode + gravity", "M2", "M3", "F")
SIMULATE = {"One-mode": simulate_one_mode, "One-mode + gravity": lambda u: simulate_one_mode(u, gravity=True),
            "M2": simulate_m2, "M3": simulate_m3, "F": simulate_f}

## 4 — Simulator, excitations and metrics

In [ ]:
def simulate_truth(cmds: np.ndarray, seed: int = 0) -> np.ndarray:
    # Simulator tip after each command step minus the settled tip, mm, (n, 3).
    sim = make_calibrated_sim(control_hz=HZ, seed=seed)
    obs = sim.reset()
    for _ in range(int(round(SETTLE_S * HZ))):
        obs = sim.step(np.full(3, BASE))
    home = np.asarray(obs["tip_pos"], float).copy()
    out = np.empty((len(cmds), 3))
    for k, c in enumerate(cmds):
        obs = sim.step(c)
        out[k] = 1000.0 * (np.asarray(obs["tip_pos"], float) - home)
    sim.close()
    return out


def excitation(kind: str, seconds: float = 60.0, seed: int = 31) -> np.ndarray:
    # Absolute commands (n, 3) for S2, S3, S4 in psi.
    n = int(round(seconds * HZ))
    if kind == "random_steps":      # independent random levels 1.5-7.5 psi, held 0.5-3 s
        g = np.random.default_rng(seed)
        out = np.empty((n, 3))
        for j in range(3):
            k = 0
            while k < n:
                hold = int(round(g.uniform(0.5, 3.0) * HZ))
                out[k:k + hold, j] = g.uniform(1.5, 7.5)
                k += hold
        return out
    if kind == "chirp":             # phased sines 0/120/240 deg, 4.5 +- 2.5 psi, 0.02 -> 1 Hz
        t = np.arange(n) * DT
        f0, f1 = 0.02, 1.0
        phase = 2 * np.pi * f0 * seconds / np.log(f1 / f0) * ((f1 / f0) ** (t / seconds) - 1)
        return BASE + 2.5 * np.sin(phase[:, None] + np.radians([0.0, 120.0, 240.0]))
    raise ValueError(kind)


def metrics(truth: np.ndarray, model: np.ndarray) -> dict:
    # Per axis: RMSE, max |error|, bias = mean error (mm), fit = 100 (1 - |e| / |y - mean y|) %.
    out = {}
    for i, ax in enumerate("xyz"):
        if np.isnan(model[:, i]).all():
            out[ax] = None
            continue
        e = model[:, i] - truth[:, i]
        out[ax] = dict(rmse_mm=float(np.sqrt(np.mean(e**2))), max_abs_mm=float(np.abs(e).max()),
                       bias_mm=float(e.mean()),
                       fit_pct=float(100 * (1 - np.linalg.norm(e) / np.linalg.norm(truth[:, i] - truth[:, i].mean()))))
    return out

## 5 — Choose an excitation and run

- `"random_steps"`: each input jumps to an independent random level in 1.5–7.5 psi, held 0.5–3 s.
- `"chirp"`: the same sine on all three inputs, shifted 0°, 120°, 240°, 4.5 ± 2.5 psi, swept 0.02 → 1 Hz.
- Or set `CMDS` yourself: any `(n, 3)` array of absolute psi for S2, S3, S4 at 100 Hz, within 0–9.

In [ ]:
KIND = "random_steps"     # or "chirp"
SECONDS = 60.0
SEED = 31                 # random_steps only

CMDS = excitation(KIND, SECONDS, seed=SEED)
# CMDS = np.column_stack([...])   # custom excitation

u = CMDS - BASE
t = np.arange(len(CMDS)) * DT
truth = simulate_truth(CMDS)
sims = {m: SIMULATE[m](u) for m in MODELS}
results = {m: metrics(truth, sims[m]) for m in MODELS}

print(f"{'model':<22}" + "".join(f"{a + ' RMSE (fit)':>22}" for a in "xyz"))
for m in MODELS:
    cells = [f"{v['rmse_mm']:.3f} mm ({v['fit_pct']:5.1f} %)" if v else "no output" for v in results[m].values()]
    print(f"{m:<22}" + "".join(f"{c:>22}" for c in cells))

## 6 — Tip paths: all models, then each model against the simulator

In [ ]:
def axis_panels(axs, names):
    for i, a in enumerate("xyz"):
        axs[i].plot(t, truth[:, i], **SIM_STYLE)
        for m in names:
            if not np.isnan(sims[m][:, i]).all():
                axs[i].plot(t, sims[m][:, i], **line(m, len(t)))
        axs[i].set(ylabel=f"tip {a} (mm)")


fig, axs = plt.subplots(4, 1, figsize=(WIDTH, PANEL_H * 4), sharex=True, layout="constrained")
for j, s in enumerate(("S2", "S3", "S4")):
    axs[0].plot(t, CMDS[:, j], **line(s, len(t)))
axs[0].set(ylabel="command (psi)", title=f"{KIND}: commands and tip, all models")
axs[0].legend(loc="upper left", bbox_to_anchor=(1.01, 1))
axis_panels(axs[1:], MODELS)
axs[1].legend(loc="upper left", bbox_to_anchor=(1.01, 1))
axs[-1].set_xlabel("time (s)")
plt.show()

for m in MODELS:
    fig, axs = plt.subplots(3, 1, figsize=(WIDTH, PANEL_H * 3), sharex=True, layout="constrained")
    axis_panels(axs, [m])
    for i, a in enumerate("xyz"):
        v = results[m][a]
        axs[i].set_title(f"{m} vs simulator, {a}: " + (f"RMSE {v['rmse_mm']:.3f} mm, fit {v['fit_pct']:.1f} %"
                                                      if v else "this model has no z output"))
    axs[0].legend(loc="upper right")
    axs[-1].set_xlabel("time (s)")
    plt.show()

## 7 — Error over time: M2, M3 and F

The one-mode models' errors are 30–100× larger and would hide these, so they are left out here.

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(WIDTH, PANEL_H * 3), sharex=True, layout="constrained")
for i, a in enumerate("xyz"):
    for m in ("M2", "M3", "F"):
        if not np.isnan(sims[m][:, i]).all():
            axs[i].plot(t, sims[m][:, i] - truth[:, i], **line(m, len(t)))
    axs[i].axhline(0, color="black", lw=1)
    axs[i].set(ylabel=f"{a} error (mm)")
axs[0].legend(loc="upper right")
axs[-1].set_xlabel("time (s)")
plt.show()

## 8 — The actuator: dead time and lag from a step

**What it describes.** The command u is a pressure set-point sent to a valve; the pressure p inside
that column's pouches is what bends the arm.

- **Dead time T.** A new command does not reach the pouch for T seconds; p(t) depends on u(t − T).
- **Lag τ.** The pouch fills through a flow restriction. Linearised, C ṗ = (G u(t − T) − p) / R with
  pneumatic resistance R and pouch capacitance C, so with τ = R C:

  **τ ṗ = −p + G u(t − T)**,   P(s) / U(s) = G e^(−sT) / (τ s + 1)

- **Step response** to Δu at t = 0: p = 0 for t < T, then p(t) = G Δu (1 − e^(−(t − T)/τ)).

**Identifying T, τ, G without knowing them.**
1. Step one input and record the measured pouch pressure; T = first sample outside the noise band.
2. Fit the step response above by least squares for T, τ, G (the cell below does 1 and 2).
3. Without a step: for each candidate delay N fit p[k+1] = α p[k] + β u[k − N]; keep the N with the
   smallest free-run error; τ = −Δt / ln α, G = β / (1 − α).
4. From a frequency sweep: the phase is −ωT − arctan(ωτ), so its slope at high ω gives T.
5. Without a pressure sensor the tip still shows T, but τ blends with the arm's own time constant.

In [ ]:
def identify_actuator(seg: int = 1, seconds: float = 6.0, pre_s: float = 2.0) -> dict:
    # +1 psi step on one input (0 = S2, 1 = S3, 2 = S4); measured pouch pressure of that column.
    sim = make_calibrated_sim(control_hz=HZ, seed=0)
    sim.reset()
    n_pre, p = int(round(pre_s * HZ)), []
    for k in range(int(round((SETTLE_S + pre_s + seconds) * HZ))):
        c = np.full(3, BASE)
        if k >= int(round((SETTLE_S + pre_s) * HZ)):
            c[seg] += 1.0
        obs = sim.step(c)
        col = int(np.asarray(obs["actuator_columns"])[seg])
        p.append(float(np.mean(obs["pouch_pressures"][col])))
    sim.close()
    p = np.array(p[-(n_pre + int(round(seconds * HZ))):])
    tt = (np.arange(len(p)) - n_pre) * DT                  # 0 = first sample with the new command
    p0, sd = p[:n_pre].mean(), p[:n_pre].std()
    k_first = int(np.argmax(np.abs(p[n_pre:] - p0) > 5 * sd))
    model = lambda x, T, tau, G: p0 + G * (1 - np.exp(-np.clip(x - T, 0, None) / tau))
    (T, tau, G), _ = curve_fit(model, tt[n_pre:], p[n_pre:], p0=(k_first * DT, 0.5, 1.0))
    return dict(t=tt, p=p, fit=model(tt, T, tau, G), p0=p0, first_rise_s=k_first * DT, T=T, tau=tau, G=G)


act = identify_actuator()
print(f"first rise at {act['first_rise_s']:.2f} s;  fit: T = {act['T']:.3f} s, tau = {act['tau']:.3f} s, G = {act['G']:.3f}")
fig, ax = plt.subplots(figsize=(WIDTH, 4.2), layout="constrained")
ax.plot(act["t"], act["p"], **{**SIM_STYLE, "label": "measured pouch pressure, S3 column"})
ax.plot(act["t"], act["fit"], color=ORANGE, ls=DASHED, lw=2.2, label="fit: p0 + G (1 − e^(−(t−T)/τ))")
ax.axvline(0, color="black", lw=1)
ax.axvline(act["T"], color="black", lw=1, ls=":")
ax.text(act["T"] + 0.05, act["p0"] + 0.5, f"T = {act['T']:.2f} s\nτ = {act['tau']:.2f} s\nG = {act['G']:.2f}")
ax.set(xlabel="time since the command step (s)", ylabel="pressure (psi)", xlim=(-1, 6), title="Actuator step response")
ax.legend(loc="lower right")
plt.show()

## 9 — Predicting 0.5 s ahead: M2 with and without a disturbance observer (NDO)

A disturbance observer estimates, from the measured tip, the part of the motion the model leaves
out: d in ẏ = −a y + a H p + d. With no measurement (a free run) it has nothing to estimate, so it is
scored on the job delay-aware controllers give the model: predicting the tip one dead time T ahead.

- **hold:** ŷ(t+T) = y(t)
- **M2 (Smith predictor):** ŷ(t+T) = y(t) + x_m(t) − x_m(t−T), with x_m = M2 without the dead time,
  driven by the commands already sent
- **M2 + NDO:** M2 + d̂ (1 − e^(−aT)) / a, with d̂ = z + l₁ y,  ż = l₁ (a y − a H p̂ − d̂)
  (p̂ = model pressure acting now). l₁ = 0.1 s⁻¹ is the gain used in the delay-aware NDOSMC.

The Smith predictor and the NDO also run together inside the delay-aware NDOSMC controller; this
cell only asks whether d̂ makes the 0.5 s forecast better.

In [ ]:
def predict_ahead(truth: np.ndarray, u: np.ndarray, l1: float = 0.1) -> dict:
    tau, g, nd = actuator_blocks()
    a, H = np.array(PARAMS["M2"]["a"]), np.array(PARAMS["M2"]["H"])
    O = np.zeros((3, 3))
    A = np.block([[-np.diag(a), np.diag(a) @ H], [O, -np.eye(3) / tau]])
    B = np.vstack([O, np.diag(g) / tau])
    xp = lsim(A, B, np.eye(6), u)                         # M2 without the dead time: [x_m, p_m]
    x_m, p_m = xp[:, :3], xp[:, 3:]
    n = len(truth)
    x_back = np.vstack([np.zeros((nd, 3)), x_m])[:n]      # x_m(t - T)
    p_now = np.vstack([np.zeros((nd, 3)), p_m])[:n]       # model pressure acting now
    d_hat, z = np.zeros((n, 3)), -l1 * truth[0]
    for k in range(n):
        d_hat[k] = z + l1 * truth[k]
        z = z + DT * l1 * (a * truth[k] - a * (H @ p_now[k]) - d_hat[k])
    smith = truth + x_m - x_back
    T = nd * DT
    return {"hold": truth[:n - nd], "M2": smith[:n - nd],
            "M2 + NDO": (smith + d_hat * (1 - np.exp(-a * T)) / a)[:n - nd], "d_hat": d_hat}


def prediction_metrics(truth, u) -> dict:
    nd = actuator_blocks()[2]
    target = truth[nd:]
    rm = lambda p: {a: float(np.sqrt(np.mean((p[:, i] - target[:, i]) ** 2))) for i, a in enumerate("xyz")}
    pr = predict_ahead(truth, u, 0.1)
    out = {"hold": rm(pr["hold"]), "M2": rm(pr["M2"])}
    for l1 in (0.1, 1.0):
        out[f"M2 + NDO (l1 = {l1:g})"] = rm(predict_ahead(truth, u, l1)["M2 + NDO"])
    return out


print(f"{'predictor, 0.5 s ahead':<26}" + "".join(f"{a + ' RMSE (mm)':>16}" for a in "xyz"))
for name, v in prediction_metrics(truth, u).items():
    print(f"{name:<26}" + "".join(f"{e:>16.3f}" for e in v.values()))

pred = predict_ahead(truth, u, 0.1)
nd = actuator_blocks()[2]
fig, axs = plt.subplots(3, 2, figsize=(WIDTH * 1.5, PANEL_H * 3), sharex=True, layout="constrained")
for i, a in enumerate("xyz"):
    axs[i, 0].plot(t[nd:], pred["M2"][:, i] - truth[nd:, i], color=BLUE, ls=SOLID, label="M2 (Smith prediction)")
    axs[i, 0].plot(t[nd:], pred["M2 + NDO"][:, i] - truth[nd:, i], color=ORANGE, ls=DASHED, label="M2 + NDO, l1 = 0.1")
    axs[i, 0].axhline(0, color="black", lw=1)
    axs[i, 0].set(ylabel=f"{a} error (mm)", title=f"{a}: prediction 0.5 s ahead − simulator")
    axs[i, 1].plot(t, pred["d_hat"][:, i], color=ORANGE, ls=DASHED, label="d̂")
    axs[i, 1].axhline(0, color="black", lw=1)
    axs[i, 1].set(ylabel=f"d̂ {a} (mm/s)", title=f"{a}: observer estimate d̂")
axs[0, 0].legend(loc="upper right")
for ax in axs[-1]:
    ax.set_xlabel("time (s)")
plt.show()

**Reading the result.** On both standard excitations, M2 + NDO predicts slightly worse than M2 alone.
d̂ comes out as a slow drift: the gap between where M2's equations say the tip settles and where the
simulator settles. The Smith prediction starts from the measured tip, so it already carries that
offset, and adding d̂ counts it twice. Inside the controller the same d̂ does help: the sliding-mode
law uses M2's equations directly, so the offset would leave a steady tracking error, and d̂ cancels
it, like integral action.

## 10 — Full table: every model, both excitations

Runs both standard excitations (random steps seed 31 and chirp, 60 s each) and tabulates per axis
the free-run RMSE, largest |error|, bias (mm) and fit (%), then the 0.5 s-ahead prediction errors
and the static gains (+1 psi step held 10 s from 4.5 psi). **Bold** = best in its column; – = no
output on that axis. Takes a few minutes.

In [ ]:
STATS = (("rmse_mm", "RMSE", "{:.3f}", lambda v: v), ("max_abs_mm", "max |e|", "{:.2f}", lambda v: v),
         ("bias_mm", "bias", "{:+.3f}", abs), ("fit_pct", "fit %", "{:.1f}", lambda v: -v))
CSS = ("<style>.mc td,.mc th{padding:3px 8px;border:1px solid #bbb;text-align:center}"
       ".mc td:first-child{text-align:left}</style>")

all_metrics, all_pred = {}, {}
for kind in ("random_steps", "chirp"):
    cmds = excitation(kind, 60.0, seed=31)
    tr = simulate_truth(cmds)
    all_metrics[kind] = {m: metrics(tr, SIMULATE[m](cmds - BASE)) for m in MODELS}
    all_pred[kind] = prediction_metrics(tr, cmds - BASE)

html = CSS
for kind, res in all_metrics.items():
    best = {(a, key): min(score(float(f.format(res[m][a][key]))) for m in MODELS if res[m][a])
            for a in "xyz" for key, _, f, score in STATS}
    head = "<tr><th rowspan=2>model</th>" + "".join(f"<th colspan=4>{a}</th>" for a in "xyz") + "</tr><tr>"
    head += "".join(f"<th>{lab}</th>" for _ in "xyz" for _, lab, _, _ in STATS) + "</tr>"
    rows = ""
    for m in MODELS:
        rows += f"<tr><td>{m}</td>"
        for a in "xyz":
            for key, _, f, score in STATS:
                v = res[m][a]
                if v is None:
                    rows += "<td>–</td>"
                    continue
                txt = f.format(v[key])
                rows += f"<td><b>{txt}</b></td>" if score(float(txt)) == best[a, key] else f"<td>{txt}</td>"
        rows += "</tr>"
    html += f"<h4>Free run, {kind} (mm; fit %)</h4><table class=mc>{head}{rows}</table>"

names = list(all_pred["chirp"])
head = "<tr><th>predictor, 0.5 s ahead</th>" + "".join(f"<th>{k} {a}</th>" for k in all_pred for a in "xyz") + "</tr>"
rows = ""
for n in names:
    rows += f"<tr><td>{n}</td>"
    for k in all_pred:
        for a in "xyz":
            v, b = all_pred[k][n][a], min(round(all_pred[k][x][a], 3) for x in names)
            rows += f"<td><b>{v:.3f}</b></td>" if round(v, 3) == b else f"<td>{v:.3f}</td>"
    rows += "</tr>"
html += f"<h4>Prediction RMSE (mm)</h4><table class=mc>{head}{rows}</table>"

head = "<tr><th>static gain (mm/psi)</th><th>Simulator</th>" + "".join(f"<th>{m}</th>" for m in MODELS) + "</tr>"
rows = ""
for name, seg, axis in (("x per psi on S3", 1, 0), ("y per psi on S2", 0, 1)):
    c = np.full((1000, 3), BASE)
    c[:, seg] += 1.0
    g = {"truth": simulate_truth(c)[-1, axis], **{m: SIMULATE[m](c - BASE)[-1, axis] for m in MODELS}}
    closest = min(MODELS, key=lambda m: abs(g[m] - g["truth"]))
    rows += f"<tr><td>{name}</td><td>{g['truth']:+.2f}</td>" + "".join(
        (f"<td><b>{g[m]:+.2f} ({100 * g[m] / g['truth']:.0f} %)</b></td>" if m == closest
         else f"<td>{g[m]:+.2f} ({100 * g[m] / g['truth']:.0f} %)</td>") for m in MODELS) + "</tr>"
html += f"<h4>Static gain (% of the simulator)</h4><table class=mc>{head}{rows}</table>"
display(HTML(html))